# 05 Advanced

Each code cell below is one of the eight scripts of `05_advanced/`, in order: the same code as the command line (`python 05_advanced/NN_name.py`), plus a first line setting `__file__`, which the scripts use to find `output/`. Change a cell to experiment; to change an example for good, edit its script and run `python sync_notebooks.py`, which copies the scripts into the notebooks. Each script's docstring says what it shows, what it needs and what it checks.

| # | Script | Tier | Time |
|---|---|---|---|
| 25 | collective effects (JFEL linac) | A; elegant with docker | ~150 s |
| 26 | ASTRA injector from the cathode | B | ~35 s |
| 27 | a ring: tunes, closed orbit, DA, frequency map | A | ~20 s |
| 28 | an energy-recovery linac (multipass) | A; elegant with docker | ~60 s |
| 29 | a plasma-wakefield stage (Wake-T) | A | ~10 s |
| 30 | start-to-end FEL, elegant into Genesis 4 | B | ~40 s |
| 31 | secondaries in BDSIM | C | ~60 s |
| 32 | Bmad, and Bmad multipass | C | ~20 s |

A script whose codes are missing prints `SKIP` (or skips its elegant runs) and carries on. Lines starting `XFAIL #N` are known bugs, listed in `ISSUES.md`. GPT, FBPIC and Ptarmigan are not covered: GPT needs its own licence, FBPIC a GPU, and Ptarmigan support isn't on SIMBA's main yet.

## 25 Collective effects: wakes, CSR, LSC

Every line has `csr_enable`, `lsc_enable` and `wakefield_enable`, and SIMBA passes them to each code in its own terms. **They start on (CSR and wakes) when a `.def` doesn't say**, so set all three when it matters. A code that doesn't model an effect tracks without it, with no error.

JFEL's linac, one effect at a time, compared at the undulator entrance. The wakes take out the chirp before the chicane, so $\sigma_z$ falls by ~40%. CSR in the chicane's bends grows $\varepsilon_{n,x}$ 3–5× and leaves $\varepsilon_{n,y}$ alone. The JFEL README's Ocelot CSR row is stale since SIMBA PR #67 (`ISSUES.md` #29), and the script prints it but doesn't check it.

In [ ]:
__file__ = "25_collective_effects.py"  # python and %run set this; a notebook cell does not
"""
25 - Collective effects: wakes, CSR, LSC
========================================

Shows: SIMBA's switches for collective effects, one at a time. Every line
has `csr_enable`, `lsc_enable` and `wakefield_enable`; SIMBA passes them to
each code in that code's own terms. A code that doesn't model an effect
tracks without it (no error), so check which codes model what:

    | Effect                   | Switch           | Modelled by (JFEL README)  |
    | structure wakes          | wakefield_enable | elegant, Ocelot            |
    | CSR in bends             | csr_enable       | elegant, Ocelot, ImpactX, Bmad |
    | longitudinal space charge| lsc_enable       | elegant, Ocelot            |

(3D space charge is set differently, with the line's
`charge: {space_charge_mode: 3d}`; it takes ~90 s in Ocelot here and is
left out.)

Run:    python 05_advanced/25_collective_effects.py
Tier:   A (Ocelot); B for elegant (docker).
Needs:  SIMBA, LAURA, laura-lattices (JFEL) and Ocelot; docker and
        ghcr.io/astec-stfc/simcodes-docker:master for elegant (skipped
        otherwise).
Output: output/25_collective_effects/<code>_<effect>/.

Units: emittance (normalised) and sizes in um; energy spread in %.

The line is JFEL's Lattices/jfel_linac_codes.def (as 13 and 14): the
250 pC injector beam from S02 to the undulator entrance, through the
chicane at ~120 MeV. Each run turns on one effect (or none) and the beam
is compared at the undulator entrance.

Checks, in each code:
- Wakes (the S- and X-band structures) reduce the chirp into the chicane:
  sigma_z falls by more than 30% (measured 37% in Ocelot, 42% in elegant).
- CSR in the chicane's bends: eps_nx grows more than 2x (measured 5.3x
  Ocelot, 3.1x elegant), and eps_ny, in the non-bending plane, changes
  less than 2% (measured < 0.2%).
- Each run reproduces the JFEL README's "Collective effects" table (eps_nx,
  sigma_x, sigma_z) to 2%, except Ocelot's CSR row. Same input, same code,
  so the match is to the README's rounding; 2% covers version drift.
  Ocelot's CSR row is printed only. Since SIMBA PR #67, Ocelot's CSR runs
  from the first bend to the next cavity, instead of over the whole line
  when the line had a `csr:` block, and eps_nx comes out 4.04 against the
  README's 3.31 (ISSUES.md #29). Ocelot was already the outlier: elegant
  gives 2.33, Bmad 2.00 and ImpactX 1.68 (README).
"""

import os
import shutil
import subprocess
from pathlib import Path

import numpy as np
import simba.Framework as fw
import simba.Modules.Beams as rbf
from laura_lattices import JFEL

OUT = Path(__file__).resolve().parents[1] / "output" / Path(__file__).stem
JFEL_DIR = os.path.dirname(JFEL.__file__)
IMAGE = "ghcr.io/astec-stfc/simcodes-docker:master"
ENTRANCE = "JFEL-FEL-SIM-MARK-01"
EFFECTS = {
    "none": dict(csr_enable=False, lsc_enable=False, wakefield_enable=False),
    "wakes": dict(csr_enable=False, lsc_enable=False, wakefield_enable=True),
    "csr": dict(csr_enable=True, lsc_enable=False, wakefield_enable=False),
    "lsc": dict(csr_enable=False, lsc_enable=True, wakefield_enable=False),
}
README = {  # JFEL README, "Collective effects": eps_nx, sigma_x, sigma_z [um]
    ("elegant", "wakes"): (0.762, 49.5, 48.2), ("ocelot", "wakes"): (0.759, 48.6, 55.3),
    ("elegant", "csr"): (2.33, 109.3, 81.5), ("ocelot", "csr"): (3.31, 154.8, 87.2),
    ("elegant", "lsc"): (0.798, 53.0, 80.4), ("ocelot", "lsc"): (0.871, 56.4, 82.9),
}
STALE = {("ocelot", "csr")}  # ISSUES.md #29


def docker_ready():
    return bool(shutil.which("docker")) and subprocess.run(
        ["docker", "image", "inspect", IMAGE], capture_output=True).returncode == 0


def track(code, effect):
    """JFEL's linac in `code` with one effect on; the beam's summary at the undulator entrance."""
    directory = OUT / f"{code}_{effect}"
    f = fw.Framework(master_lattice=JFEL_DIR, directory=str(directory), clean=True, verbose=False,
                     container_runtime="docker" if code == "elegant" else None)
    f.loadSettings("Lattices/jfel_linac_codes.def")
    f.change_Lattice_Code("Linac", code)
    for switch, value in EFFECTS[effect].items():
        setattr(f["Linac"], switch, value)
    f.track(files=["Linac"], check_lattice=False)
    beam = rbf.load_file(str(directory / f"{ENTRANCE}.openpmd.hdf5"))
    cp = beam.cp.val
    return {"eps_nx": beam.Particles.emittance.enx.val * 1e6, "eps_ny": beam.Particles.emittance.eny.val * 1e6,
            "sigma_x": np.std(beam.x.val) * 1e6, "sigma_z": np.std(beam.z.val) * 1e6,
            "dE": np.std(cp) / np.mean(cp) * 100}


codes = ["ocelot"] + (["elegant"] if docker_ready() else [])
if "elegant" not in codes:
    print(f"elegant skipped: docker or {IMAGE} not available\n")
results = {(code, effect): track(code, effect) for code in codes for effect in EFFECTS}

print(f"{'code':8s} {'effect':6s} {'eps_nx':>7s} {'eps_ny':>7s} {'sigma_x':>8s} {'sigma_z':>8s} {'dE [%]':>7s}   README (eps_nx, sigma_x, sigma_z)")
for (code, effect), r in results.items():
    ref = README.get((code, effect))
    note = "" if ref is None else f"   {ref}" + ("  stale, ISSUES.md #29" if (code, effect) in STALE else "")
    print(f"{code:8s} {effect:6s} {r['eps_nx']:7.3f} {r['eps_ny']:7.3f} {r['sigma_x']:8.1f} {r['sigma_z']:8.1f} {r['dE']:7.3f}{note}")

# --- Checks ----------------------------------------------------------------
problems = []
for code in codes:
    none, wakes, csr = (results[(code, e)] for e in ("none", "wakes", "csr"))
    if wakes["sigma_z"] / none["sigma_z"] > 0.7:
        problems.append(f"{code}: wakes changed sigma_z only {wakes['sigma_z'] / none['sigma_z']:.2f}x")
    if csr["eps_nx"] / none["eps_nx"] < 2:
        problems.append(f"{code}: CSR grew eps_nx only {csr['eps_nx'] / none['eps_nx']:.2f}x")
    if abs(csr["eps_ny"] / none["eps_ny"] - 1) > 0.02:
        problems.append(f"{code}: CSR changed eps_ny by {csr['eps_ny'] / none['eps_ny'] - 1:+.1%}")
for key, ref in README.items():
    if key in results and key not in STALE:
        got = [results[key][q] for q in ("eps_nx", "sigma_x", "sigma_z")]
        worst = max(abs(g / r - 1) for g, r in zip(got, ref))
        if worst > 0.02:
            problems.append(f"{key}: {np.round(got, 3)} against the README's {ref}")
assert not problems, "\n".join(problems)
print(f"\nOK ({', '.join(codes)}): wakes shorten the bunch, CSR grows eps_nx only, and the README's table is reproduced")

## 26 The injector, from the cathode (ASTRA)

JFEL's ASTRA injector: the generator makes the cathode distribution, `setSeedValue` fixes its random numbers, and ASTRA tracks it to S02, ~35 MeV, 250 pC. The result is compared with the beam laura-lattices saves as `Data_Files/jfel_injector_S02`, which the other JFEL examples start from. At that file's 32768 particles it is reproduced exactly. The default here is 4096, for speed, and then the emittance is within 15% (sampling); `python 26_injector_from_cathode.py --particles 32768` takes ~5 min.

In [ ]:
__file__ = "26_injector_from_cathode.py"  # python and %run set this; a notebook cell does not
"""
26 - Track the JFEL injector from the cathode in ASTRA
======================================================

Shows: a start-to-end run's first stage, from the cathode. SIMBA generates
the laser-shaped bunch at the cathode (the .def's `generator:` block, with
ASTRA's generator and JFEL's jfel.yaml defaults), then ASTRA tracks it
through the HRRG gun and the first linac (L01) to ~35 MeV, with 2D space
charge and the cathode's mirror charge. This is how JFEL's saved injector
beam, Data_Files/jfel_injector_S02.openpmd.hdf5, the input to 13, 14, 22,
23 and 25, was made (JFEL's Generators/make_injector_beam.py).

    framework = fw.Framework(master_lattice=JFEL + "/", generator_defaults="/jfel.yaml", ...)
    framework.loadSettings("Lattices/jfel_combined.def")
    framework["generator"].number_of_particles = 4096
    framework.setSeedValue(1234)
    framework.track(endfile="injector400")   # run up to and including this line

Run:    python 05_advanced/26_injector_from_cathode.py [--particles 4096]
Tier:   B
Needs:  SIMBA, LAURA, laura-lattices (JFEL), docker and
        ghcr.io/astec-stfc/simcodes-docker:master (ASTRA).
Output: output/26_injector_from_cathode/: ASTRA's files and the beam at
        every screen.

Units: energy in MeV; charge in pC; emittance (normalised) in um; sigma_t
in ps; sigma_x in mm.

Checks, against the saved beam (32768 particles, seed 1234; the README:
"250 pC, 35 MeV, eps_n 0.81 um, 3 ps flat-top"):
- The energy to 0.1%, the charge to 1e-9, sigma_t to 2% and sigma_x to 3%
  (measured at 4096: 5e-5, exact, 0.1%, 0.3%; sampling noise ~1/sqrt(N)
  is 1.6% for a size).
- eps_n to 15% with fewer than 16384 particles, 2% with more. Space charge
  with few macroparticles adds numerical emittance growth, more than the
  sampling noise: measured +10% at 4096 (26 s), +0.6% at 16384 (2 min).
- With 32768 particles (5 min), the same seed gives back the saved beam
  exactly (measured 1e-13): the run is reproducible.
"""

import argparse
import os
import shutil
import subprocess
from pathlib import Path

import numpy as np
import simba.Framework as fw
import simba.Modules.Beams as rbf
from laura_lattices import JFEL

OUT = Path(__file__).resolve().parents[1] / "output" / Path(__file__).stem
JFEL_DIR = os.path.dirname(JFEL.__file__)
IMAGE = "ghcr.io/astec-stfc/simcodes-docker:master"
SCREEN = "JFEL-S02-SIM-APER-01"
SAVED_PARTICLES, SEED = 32768, 1234

parser = argparse.ArgumentParser()
parser.add_argument("--particles", type=int, default=4096)
args = parser.parse_args([])  # no command line here: options go in the []


def summary(beam):
    q = np.abs(beam.Particles.charge.val)
    return {"E [MeV]": np.mean(np.hypot(beam.cp.val, 0.51099895e6)) / 1e6, "Q [pC]": q.sum() * 1e12,
            "eps_nx [um]": beam.Particles.emittance.enx.val * 1e6, "eps_ny [um]": beam.Particles.emittance.eny.val * 1e6,
            "sigma_t [ps]": np.std(beam.t.val) * 1e12, "sigma_x [mm]": np.std(beam.x.val) * 1e3}


if not (shutil.which("docker") and subprocess.run(["docker", "image", "inspect", IMAGE],
                                                  capture_output=True).returncode == 0):
    print(f"SKIP: needs docker and {IMAGE} for ASTRA")
else:  # not sys.exit(0): in a notebook that is an error and stops "Run All"
    framework = fw.Framework(master_lattice=JFEL_DIR + "/", generator_defaults="/jfel.yaml", directory=str(OUT),
                             clean=True, verbose=False, container_runtime="docker")
    framework.loadSettings("Lattices/jfel_combined.def")
    framework["generator"].number_of_particles = args.particles
    framework.setSeedValue(SEED)
    framework.track(endfile="injector400")

    run = summary(rbf.load_file(str(OUT / f"{SCREEN}.openpmd.hdf5")))
    saved = summary(rbf.load_file(os.path.join(JFEL_DIR, "Data_Files", "jfel_injector_S02.openpmd.hdf5")))
    print(f"\nAt {SCREEN}:")
    print(f"{'':14s} {f'this run ({args.particles})':>18s} {f'saved ({SAVED_PARTICLES})':>15s} {'rel diff':>9s}")
    for key in run:
        print(f"{key:14s} {run[key]:18.4f} {saved[key]:15.4f} {run[key] / saved[key] - 1:9.1e}")

    # --- Checks ------------------------------------------------------------
    eps_tol = 0.15 if args.particles < 16384 else 0.02
    limits = {"E [MeV]": 1e-3, "Q [pC]": 1e-9, "sigma_t [ps]": 0.02, "sigma_x [mm]": 0.03,
              "eps_nx [um]": eps_tol, "eps_ny [um]": eps_tol}
    if args.particles == SAVED_PARTICLES:
        limits = dict.fromkeys(limits, 1e-9)
    bad = [f"{k}: {run[k]:.4f} against {saved[k]:.4f}" for k, tol in limits.items() if abs(run[k] / saved[k] - 1) > tol]
    assert not bad, "\n".join(bad)
    print(f"\nOK: {run['E [MeV]']:.1f} MeV, {run['Q [pC]']:.0f} pC, eps_n {run['eps_nx [um]']:.2f} um from the cathode"
          + (" (the saved beam, exactly)" if args.particles == SAVED_PARTICLES else ""))

## 27 A ring: tunes, closed orbit, dynamic aperture, frequency map

A LAURA section with `geometry: closed` is a ring. `tracking: {periodic: True}` and `ring_parameters()` give the tunes, periodic Twiss and closed orbit; `tracking: {turns, dynamic_aperture}` with `run_dynamic_aperture()` and `run_frequency_map()` track many turns. The ring is SIMBA's own `examples/ring_studies` FODO ring (`ring.py`), and the reference is an independent symplectic tracker in `ring.py`.

The tunes agree everywhere, to $10^{-9}$ with the matrices. The rest has known bugs: Xsuite finds no closed orbit without RF (#30), Ocelot reports a zero orbit (#31), and Ocelot and MAD-X give a dynamic aperture ~30% smaller than Xsuite, elegant and the tracker (#32).

In [ ]:
__file__ = "27_ring_tune_da.py"  # python and %run set this; a notebook cell does not
"""
27 - A ring: tunes, closed orbit, dynamic aperture, frequency map
=================================================================

Shows: SIMBA's ring studies. A LAURA section with `geometry: closed` is a
ring. Then, on its line:
- `tracking: {periodic: True}`, track(), and `ring_parameters()` give the
  tunes, periodic Twiss, chromaticity and the closed orbit;
- `tracking: {turns: N, dynamic_aperture: {...}}` and
  `run_dynamic_aperture()` track starts along rays from the origin (as
  elegant's find_aperture) and return (x, y, turns survived);
  `dynamic_aperture_boundary()` gives the last survivor on each ray;
- `run_frequency_map()` gives (x, y, Qx, Qy, diffusion) over a grid.

Run:    python 05_advanced/27_ring_tune_da.py
Tier:   A
Needs:  SIMBA, LAURA, Ocelot, Xsuite, MAD-X (cpymad); nafflib for the
        frequency map's tunes.
Output: output/27_ring_tune_da/<code>/ and dynamic_aperture.png.

Units: tunes in turns; positions in mm; k1l in 1/m, k2l in 1/m^2.

The ring is SIMBA's examples/ring_studies/da_fma.py (05_advanced/ring.py):
ten 4 m FODO cells, a sextupole beside each quad, 1 GeV, no RF. The
reference for the nonlinear checks is ring.py's own tracker: exact
thick-quad matrices and sliced thin sextupoles, symplectic, independent of
SIMBA and the codes.

Checks:
- Tunes: each code's equals the thick-quad matrix tune to 1e-6 (the
  sextupoles do nothing on axis). Measured: identical to 1e-9.
- Closed orbit with QF0 displaced 1 mm: MAD-X's equals the tracker's to
  1e-3 (measured 2.4e-4; MAD-X tracks thin slices). The linear formula
  (I - M)^-1 kick, printed, is 7% off: off axis the sextupoles add
  focusing. XFAIL #30: Xsuite finds no closed orbit (its 6D search, in a
  ring without RF), and so gives no optics at all; `line.twiss(method="4d")`
  gets the tracker's orbit to 4e-7. XFAIL #31: Ocelot reports a zero orbit.
- Dynamic aperture on the +x ray, 512 turns: Xsuite's equals the
  tracker's to one grid step (1.03 mm); measured 16.55 against 16.55 (the
  tracker gives 15.5-16.6 with 4 to 50 slices). elegant agrees too (SIMBA's
  own da_fma.png). XFAIL #32: Ocelot gives 12.4 mm and MAD-X 11.4. For
  Ocelot that's its second-order maps (SecondTM, not symplectic), which
  LAURA chooses for the sextupoles: with kick maps it gives the tracker's
  one-kick value. For MAD-X the cause isn't found; more slices don't
  change it.
- Frequency map: each code's tune at the smallest amplitude (1 mm) equals
  the linear tune to 1e-3 (measured 2-4e-4, the amplitude detuning).
"""

import sys
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parent))
sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "02_tracking"))
import ring  # noqa: E402
from simple_lattice import transfer_matrix  # noqa: E402

OUT = Path(__file__).resolve().parents[1] / "output" / Path(__file__).stem
CODES = ["ocelot", "xsuite", "madx"]
TURNS, OFFSET = 512, 1e-3
DA = {"nx": 30, "ny": 15, "n_lines": 11, "x_max": 0.03, "y_max": 0.015}
X_RAY = DA["x_max"] * np.arange(1, DA["nx"]) / (DA["nx"] - 1)  # the +x ray's starts
STEP = X_RAY[0]


def matrix_tune(plane):
    m = transfer_matrix(ring.elements(), plane, 0.0, ring.CELLS * ring.CELL_LENGTH)
    mu = np.arccos(np.trace(m) / 2)
    return (2 * np.pi - mu if m[0, 1] < 0 else mu) / (2 * np.pi)


def linear_orbit():
    """(I - M)^-1 times QF0's kick, sextupoles off."""
    quads = [e for e in ring.elements() if type(e).__name__ == "Quadrupole"]
    qf0 = quads[0]
    start, end = qf0.physical.middle.z - qf0.physical.length / 2, qf0.physical.middle.z + qf0.physical.length / 2
    m_q = transfer_matrix([qf0], "x", start, end)
    rest = transfer_matrix(quads[1:], "x", end, ring.CELLS * ring.CELL_LENGTH)
    m = rest @ m_q @ transfer_matrix([], "x", 0.0, start)
    kick = rest @ (np.array([OFFSET, 0.0]) - m_q @ np.array([OFFSET, 0.0]))  # u -> M_q (u - d) + d
    return np.linalg.solve(np.eye(2) - m, kick)


tunes = {"matrix": (matrix_tune("x"), matrix_tune("y"))}
orbits, apertures, boundaries, small = {}, {}, {}, {}
for code in CODES:
    f = ring.framework_for(code, OUT / code / "optics", {"periodic": True})
    f.track()
    p = f["RING"].ring_parameters()
    tunes[code] = (p["tune_x"], p["tune_y"])

    f = ring.framework_for(code, OUT / code / "orbit", {"periodic": True})
    f.modifyElement("QF0", "physical.error.position.x", OFFSET)
    f.track()
    orbits[code] = f["RING"].ring_parameters().get("closed_orbit_x")

    f = ring.framework_for(code, OUT / code / "aperture", {"turns": TURNS, "dynamic_aperture": DA})
    line = f["RING"]
    line.preProcess()
    line.write()
    boundaries[code] = line.dynamic_aperture_boundary(line.run_dynamic_aperture())
    apertures[code] = boundaries[code][0][0]  # the first ray is +x
    footprint = line.run_frequency_map()
    small[code] = min(footprint, key=lambda r: r[0] ** 2 + r[1] ** 2)[2:4]

reference_orbit = ring.closed_orbit(OFFSET)[0]
reference_aperture = ring.on_axis_aperture(X_RAY, TURNS)

print(f"\n{'':8s} {'tune_x':>10s} {'tune_y':>10s}   {'orbit x [mm]':>12s}   {'DA +x [mm]':>10s}   {'FMA tunes at 1 mm':>18s}")
for name, (qx, qy) in tunes.items():
    if name == "matrix":
        print(f"{'matrix':8s} {qx:10.7f} {qy:10.7f}   {linear_orbit()[0] * 1e3:12.5f}   {'':10s}   (linear formula)")
        continue
    orbit = "none" if orbits[name] is None else f"{orbits[name] * 1e3:.5f}"
    print(f"{name:8s} {qx:10.7f} {qy:10.7f}   {orbit:>12s}   {apertures[name] * 1e3:10.2f}   "
          f"{small[name][0]:.5f} {small[name][1]:.5f}")
print(f"{'tracker':8s} {'':10s} {'':10s}   {reference_orbit * 1e3:12.5f}   {reference_aperture * 1e3:10.2f}")

plt.figure(figsize=(7, 4))
for code in CODES:
    xy = np.array(boundaries[code]) * 1e3
    plt.plot(xy[:, 0], xy[:, 1], "o-", label=code)
plt.plot([reference_aperture * 1e3], [0], "k*", markersize=12, label="independent tracker (+x)")
plt.xlabel("x [mm]")
plt.ylabel("y [mm]")
plt.title(f"Dynamic aperture, {TURNS} turns")
plt.legend()
plt.savefig(OUT / "dynamic_aperture.png", dpi=120, bbox_inches="tight")
print(f"\nPlot: {OUT / 'dynamic_aperture.png'}\n")

# --- Checks ----------------------------------------------------------------
problems = []
for code in CODES:
    if max(abs(a - b) for a, b in zip(tunes[code], tunes["matrix"])) > 1e-6:
        problems.append(f"{code}: tunes {tunes[code]}, matrices {tunes['matrix']}")
    if max(abs(a - b) for a, b in zip(small[code], tunes["matrix"])) > 1e-3:
        problems.append(f"{code}: frequency-map tunes at 1 mm {small[code]}, linear {tunes['matrix']}")

if abs(orbits["madx"] / reference_orbit - 1) > 1e-3:
    problems.append(f"madx: closed orbit {orbits['madx']}, tracker {reference_orbit}")
if orbits["xsuite"] is None:
    print("XFAIL #30: Xsuite found no closed orbit (6D search without RF), so no optics with the offset quad")
elif abs(orbits["xsuite"] / reference_orbit - 1) < 1e-3:
    print("XPASS #30: Xsuite finds the closed orbit; check it here and update ISSUES.md")
if orbits["ocelot"] == 0.0:
    print("XFAIL #31: Ocelot reports a zero closed orbit with QF0 displaced 1 mm")
elif abs(orbits["ocelot"] / reference_orbit - 1) < 1e-3:
    print("XPASS #31: Ocelot's closed orbit is right; check it here and update ISSUES.md")

if abs(apertures["xsuite"] - reference_aperture) > STEP * 1.01:
    problems.append(f"xsuite: DA {apertures['xsuite']:.4f}, tracker {reference_aperture:.4f}")
for code in ("ocelot", "madx"):
    if abs(apertures[code] - reference_aperture) > STEP * 1.01:
        print(f"XFAIL #32: {code}'s dynamic aperture on +x is {apertures[code] * 1e3:.2f} mm, "
              f"the tracker's {reference_aperture * 1e3:.2f}")
    else:
        print(f"XPASS #32: {code}'s dynamic aperture now agrees; update ISSUES.md")
assert not problems, "\n".join(problems)
print("OK: tunes from every code, MAD-X's closed orbit and Xsuite's dynamic aperture match the references")

In [ ]:
from IPython.display import Image, display
display(Image(filename="../output/27_ring_tune_da/dynamic_aperture.png"))

## 28 An energy-recovery linac: one cryomodule, two passes

A LAURA multipass layout: the beam crosses the same eight cavities twice. In a line the linac's elements appear once per pass, `NAME.1` and `NAME.2`, and the second pass's phase is set on the `.2` cavities alone. The beam goes 10 → 100 → 10 MeV/c, the RF takes back all the energy it gave, and the dump beam is the injected beam again: the racetrack is mirror-symmetric.

At the dump screen $\varepsilon_{n,x}$ reads 5 µm. That is the dump dipole's dispersion ($D = -0.58$ m) times the spread, not growth. Take out the $x$–$\delta$ correlation and it is 1.0 µm again. A path-length error, as a phase offset $\phi$ on the return pass, raises the dump momentum as $10 + 90\,(1 - \cos\phi)$ MeV/c.

In [ ]:
__file__ = "28_erl_multipass.py"  # python and %run set this; a notebook cell does not
"""
28 - An energy-recovery linac: one cryomodule, two passes
=========================================================

Shows: a LAURA multipass layout in SIMBA. laura-lattices' ERL crosses the
same cavities twice: 10 -> 100 MeV/c on crest, round a racetrack, then back
through them half an RF period late, 100 -> 10 MeV/c, handing the energy
back to the RF. The layout names the linac's elements per pass, `NAME#1`
and `NAME#2`; in a tracking line (and its beam files) they are `NAME.1` and
`NAME.2`. The second pass's cavity phase (180 deg) belongs to the pass, so
a path-length error is a change to the `.2` cavities only:

    for name, el in f["ERL"].elements.items():
        if name.startswith("ERL-LIN-RF-CAV") and name.endswith(".2"):
            el.cavity.phase = 180 + offset

Run:    python 05_advanced/28_erl_multipass.py
Tier:   A (Ocelot); B for elegant (docker).
Needs:  SIMBA and LAURA with multipass layouts, laura-lattices (ERL) and
        Ocelot; docker and ghcr.io/astec-stfc/simcodes-docker:master for
        elegant (skipped otherwise).
Output: output/28_erl_multipass/<code>_<offset>/.

Units: momentum in MeV/c; spread relative, rms; emittance (normalised) in
um; sizes in mm; sigma_t in ps; phase offset in degrees.

The lattice is ERL/Lattices/erl.def: line ERL (injector to the linac exit
on pass 2) and line DUMP. No CSR or synchrotron radiation. Each code runs
at offset 0; Ocelot also at +-5, +-10, +-20 deg, elegant at +10 (18 s a run).

Checks:
- Offset 0: the momentum is 10, 100, 10 MeV/c at injection, the top and the
  dump, to 1e-4 (measured 1.2e-5, elegant at the top); the RF takes back
  all the energy it gave to 1e-4 (measured 5e-6).
- Offset 0, at the linac exit on pass 2: the beam is the injected beam
  again (the racetrack is mirror-symmetric). eps_nx, eps_ny, sigma_x,
  sigma_y and sigma_t equal the injected ones to 1% (measured 0.1% or
  less).
- At the dump screen, after the dump dipole, eps_nx reads 5 um: that is
  dispersion (D = -0.58 m) times the spread, not growth. Taking out the
  x-delta correlation gives the injected eps_nx to 1% (measured 0.998).
- Offset 0: the ERL README's single-particle table (top and dump momentum
  and spread, dump sizes and sigma_t), to 1%; same input and code, so this
  is the README's rounding plus version drift.
- Phase offset: the dump momentum is 10 + 90 (1 - cos offset) MeV/c to
  0.05 MeV/c. The formula ignores the bunch's phase spread and its
  velocity (the README's elegant scan spans 15.40-15.46 at +-20 deg).
  Measured: Ocelot within 0.007, elegant 0.019 low at +10. The spread is
  90 sin(offset) x the RF phase spread (2 pi 1.3 GHz sigma_t = 0.016 rad),
  with the injected 0.01 MeV/c in quadrature, relative to the dump
  momentum, to 3% (measured 1.2% or less).
"""

import os
import shutil
import subprocess
from pathlib import Path

import numpy as np
import simba.Framework as fw
import simba.Modules.Beams as rbf
from laura_lattices import ERL

OUT = Path(__file__).resolve().parents[1] / "output" / Path(__file__).stem
ERL_DIR = os.path.dirname(ERL.__file__)
IMAGE = "ghcr.io/astec-stfc/simcodes-docker:master"
MASS = 0.51099895  # MeV/c^2
POINTS = {"injected": "ERL-LIN-SIM-START-01.1", "top": "ERL-LIN-SIM-END-01.1",
          "dump": "ERL-LIN-SIM-END-01.2", "screen": "ERL-DMP-DIA-SCR-01"}
OFFSETS = {"ocelot": [0, -20, -10, -5, 5, 10, 20], "elegant": [0, 10]}
GAIN, RF = 90.0, 1.3e9  # MeV/c per pass; Hz
README = {  # ERL README, single-particle: top p, spread; dump p, spread, sigma_x, sigma_y [mm], sigma_t [ps]
    "elegant": (99.999, 2.00e-4, 10.000, 1.01e-3, 0.581, 0.581, 2.01),
    "ocelot": (100.000, 1.98e-4, 10.000, 1.00e-3, 0.580, 0.581, 2.01),
}


def docker_ready():
    return bool(shutil.which("docker")) and subprocess.run(
        ["docker", "image", "inspect", IMAGE], capture_output=True).returncode == 0


def summary(beam):
    cp, d = beam.cp.val / 1e6, beam.cp.val / np.mean(beam.cp.val) - 1
    x, xp = beam.x.val, beam.xp.val
    dx, dxp = np.polyfit(d, x, 1)[0], np.polyfit(d, xp, 1)[0]
    eps_dispersion_free = np.sqrt(np.linalg.det(np.cov(x - dx * d, xp - dxp * d))) * np.mean(cp) / MASS
    return {"p": np.mean(cp), "spread": np.std(d), "eps_nx": beam.Particles.emittance.enx.val * 1e6,
            "eps_ny": beam.Particles.emittance.eny.val * 1e6, "sigma_x": np.std(x) * 1e3,
            "sigma_y": np.std(beam.y.val) * 1e3, "sigma_t": np.std(beam.t.val) * 1e12,
            "eps_nx_no_D": eps_dispersion_free * 1e6, "D": dx}


def track(code, offset):
    """The ERL in `code`, return-pass phase 180 + offset; a summary at each of POINTS."""
    directory = OUT / f"{code}_{offset:+d}"
    f = fw.Framework(master_lattice=ERL_DIR + "/", directory=str(directory), clean=True, verbose=False,
                     container_runtime="docker" if code == "elegant" else None)
    f.loadSettings("Lattices/erl.def")
    for line in ("ERL", "DUMP"):
        f.change_Lattice_Code(line, code)
        f[line].csr_enable = False
    for name, el in f["ERL"].elements.items():
        if name.startswith("ERL-LIN-RF-CAV") and name.endswith(".2"):
            el.cavity.phase = 180.0 + offset
    f.track(check_lattice=False)
    return {k: summary(rbf.load_file(str(directory / f"{p}.openpmd.hdf5"))) for k, p in POINTS.items()}


def energy(p):
    return np.hypot(p, MASS)


codes = ["ocelot"] + (["elegant"] if docker_ready() else [])
if "elegant" not in codes:
    print(f"elegant skipped: docker or {IMAGE} not available\n")
runs = {(code, offset): track(code, offset) for code in codes for offset in OFFSETS[code]}

print(f"{'code':8s} {'point':9s} {'p':>9s} {'spread':>9s} {'eps_nx':>7s} {'eps_ny':>7s} {'sig_x':>7s} {'sig_y':>7s} {'sig_t':>6s}   eps_nx without D")
for code in codes:
    for point, s in runs[(code, 0)].items():
        print(f"{code:8s} {point:9s} {s['p']:9.4f} {s['spread']:9.3e} {s['eps_nx']:7.3f} {s['eps_ny']:7.3f} "
              f"{s['sigma_x']:7.4f} {s['sigma_y']:7.4f} {s['sigma_t']:6.3f}   {s['eps_nx_no_D']:.3f} (D {s['D']:+.3f} m)")
    r = runs[(code, 0)]
    recovered = (energy(r["top"]["p"]) - energy(r["dump"]["p"])) / (energy(r["top"]["p"]) - energy(r["injected"]["p"]))
    print(f"{code:8s} energy taken back by the RF: {recovered:.5%}\n")


def law(offset, sigma_t):
    p = 10.0 + GAIN * (1 - np.cos(np.radians(offset)))
    spread = np.hypot(GAIN * np.sin(np.radians(offset)) * 2 * np.pi * RF * sigma_t * 1e-12, 10.0 * 1e-3) / p
    return p, spread


print(f"{'code':8s} {'offset':>6s} {'dump p':>8s} {'formula':>8s} {'spread':>9s} {'formula':>9s}")
for (code, offset), r in runs.items():
    p, spread = law(offset, r["injected"]["sigma_t"])
    print(f"{code:8s} {offset:+6d} {r['dump']['p']:8.4f} {p:8.4f} {r['dump']['spread']:9.3e} {spread:9.3e}")

# --- Checks ----------------------------------------------------------------
problems = []
for code in codes:
    r = runs[(code, 0)]
    for point, target in (("injected", 10.0), ("top", 100.0), ("dump", 10.0)):
        if abs(r[point]["p"] / target - 1) > 1e-4:
            problems.append(f"{code}: {point} momentum {r[point]['p']:.5f}, expected {target}")
    recovered = (energy(r["top"]["p"]) - energy(r["dump"]["p"])) / (energy(r["top"]["p"]) - energy(r["injected"]["p"]))
    if abs(recovered - 1) > 1e-4:
        problems.append(f"{code}: energy recovered {recovered:.5%}")
    for q in ("eps_nx", "eps_ny", "sigma_x", "sigma_y", "sigma_t"):
        if abs(r["dump"][q] / r["injected"][q] - 1) > 0.01:
            problems.append(f"{code}: dump {q} {r['dump'][q]:.4f}, injected {r['injected'][q]:.4f}")
    if abs(r["screen"]["eps_nx_no_D"] / r["injected"]["eps_nx"] - 1) > 0.01:
        problems.append(f"{code}: eps_nx at the screen without dispersion {r['screen']['eps_nx_no_D']:.3f}")
    got = (r["top"]["p"], r["top"]["spread"], r["dump"]["p"], r["dump"]["spread"],
           r["dump"]["sigma_x"], r["dump"]["sigma_y"], r["dump"]["sigma_t"])
    if max(abs(g / ref - 1) for g, ref in zip(got, README[code])) > 0.01:
        problems.append(f"{code}: {np.round(got, 5)} against the README's {README[code]}")
for (code, offset), r in runs.items():
    p, spread = law(offset, r["injected"]["sigma_t"])
    if abs(r["dump"]["p"] - p) > 0.05:
        problems.append(f"{code} {offset:+d} deg: dump momentum {r['dump']['p']:.4f}, formula {p:.4f}")
    if abs(r["dump"]["spread"] / spread - 1) > 0.03:
        problems.append(f"{code} {offset:+d} deg: dump spread {r['dump']['spread']:.3e}, formula {spread:.3e}")
assert not problems, "\n".join(problems)
print(f"\nOK ({', '.join(codes)}): 10 -> 100 -> 10 MeV/c, all the energy recovered, the injected beam back at "
      "the dump, and the phase-offset law")

## 29 A plasma-wakefield stage in Wake-T

A LAURA `Plasma` element, without a laser, is a beam-driven stage. A 1 GeV driver gives up about 15% of what it loses to a 100 MeV witness 80 µm behind it, which reaches ~325 MeV in 1 cm. The rest stays in the wake. `f.change_Lattice_Code("PWFA", "fbpic")` would run the same element in FBPIC (GPU). With 2 mm matching ramps either side, $\beta$ at the exit grows 5×, and the witness's emittance grows less in the drift that follows.

In [ ]:
__file__ = "29_plasma_wake_t.py"  # python and %run set this; a notebook cell does not
"""
29 - A plasma-wakefield stage in Wake-T
=======================================

Shows: a plasma stage as one more SIMBA line. A LAURA `Plasma` element
holds the density profile and the simulation box; with no laser the
incoming beam drives the wake (PWFA). The stage runs in Wake-T
(quasi-static, seconds on a CPU) or, with one call, in FBPIC (full PIC,
GPU; not run here):

    f.change_Lattice_Code("PWFA", "fbpic")

The next line (Ocelot) carries the beam on through a 0.5 m drift to a
screen.

Run:    python 05_advanced/29_plasma_wake_t.py
Tier:   A
Needs:  SIMBA, LAURA, laura-lattices (PLASMA), Wake-T and Ocelot (all in
        simba-accelerator[simcodes]).
Output: output/29_plasma_wake_t/<lattice>/.

Units: gamma (Lorentz factor); spread relative, rms; emittance normalised,
in um, against u_x = p_x / mc as Wake-T reports it; beta in mm; energy in mJ.

Two of laura-lattices' PLASMA lattices, each a 500 pC, 1 GeV driver and a
50 pC, 100 MeV witness 80 um behind it, through 1 cm of plasma at 1e23 m^-3:
- pwfa_single: the plateau alone;
- pwfa_ramps: the same plateau between 2 mm matching ramps (density n0/25
  at the ends), so beta grows as n^-1/2 on the way out.
The driver and witness are told apart by energy (500 MeV), as in
PLASMA/Scripts/run_example.py.

Checks:
- The driver loses energy and the witness gains it: in pwfa_single the
  driver's gamma falls from 2000 to 1702 and the witness's rises from 200
  to 637. The witness gains less energy than the driver loses (measured
  15%, 14% with the ramps; the rest stays in the wake).
- Each stage reproduces the PLASMA README's witness table (gamma, spread,
  eps_nx and beta at the exit, eps_nx at the screen) to 1%. Same input and
  codes, so this is rounding plus version drift. Measured: 0.3% or less
  (pwfa_single's screen eps_nx is 1.994 against 2.000).
- The ramps do what they are for: beta at the exit grows more than 4x
  (designed 5x, measured 5.1x), and the emittance growth in the drift to
  the screen is smaller (measured 1.83 against 1.99 um).
"""

import os
from pathlib import Path

import numpy as np
import simba.Framework as fw
import simba.Modules.Beams as rbf
from laura_lattices import PLASMA

OUT = Path(__file__).resolve().parents[1] / "output" / Path(__file__).stem
PLASMA_DIR = os.path.dirname(PLASMA.__file__)
MC2, E = 0.51099895e6, 1.602176634e-19  # eV; C
SPLIT = 500e6  # eV/c: the driver above, the witness below
LATTICES = {"pwfa_single": "PWFA", "pwfa_ramps": "PWFAR"}  # lattice: its plasma line
README = {  # PLASMA README, pwfa_ramps table: witness gamma, spread, eps_nx at exit, beta at exit [mm], eps_nx at screen
    "pwfa_single": (636.7, 0.0071, 0.987, 0.59, 2.000),
    "pwfa_ramps": (648.4, 0.0079, 0.991, 3.01, 1.829),
}


def bunch(beam, witness):
    """gamma, spread, eps_nx [um], beta [mm], energy [mJ] of the driver or the witness."""
    sel = (beam.cp.val <= SPLIT) if witness else (beam.cp.val > SPLIT)
    q = np.abs(np.asarray(beam.Particles.charge.val))[sel]
    gamma = np.hypot(beam.cp.val[sel] / MC2, 1)
    mean = np.average(gamma, weights=q)
    cov = np.cov(np.vstack([beam.x.val[sel], beam.cpx.val[sel] / MC2]), aweights=q)
    eps = np.sqrt(np.linalg.det(cov))
    return {"gamma": mean, "spread": np.sqrt(np.cov(gamma, aweights=q)) / mean, "eps_nx": eps * 1e6,
            "beta": cov[0, 0] / (eps / mean) * 1e3, "energy": np.sum(q * gamma) * MC2 * 1e3}  # q/e * gamma mc^2 [eV] * e


def run(lattice):
    """The stage and its drift; driver and witness at the entrance, the plasma's exit and the screen."""
    directory = OUT / lattice
    f = fw.Framework(master_lattice=PLASMA_DIR + "/", directory=str(directory), clean=True, verbose=False)
    f.loadSettings(f"Lattices/{lattice}.def")
    f.track()
    plasma, drift = f[LATTICES[lattice]], f[LATTICES[lattice] + "_DIAG"]
    beams = {"entrance": rbf.load_file(f"{PLASMA_DIR}/Data_Files/{lattice.replace('_single', '')}_driver_witness.openpmd.hdf5"),
             "exit": rbf.load_file(str(directory / f"{plasma.end}.openpmd.hdf5")),
             "screen": rbf.load_file(str(directory / f"{drift.end}.openpmd.hdf5"))}
    return {(where, who): bunch(b, who == "witness") for where, b in beams.items() for who in ("driver", "witness")}


results = {lattice: run(lattice) for lattice in LATTICES}

print(f"{'lattice':12s} {'where':9s} {'bunch':8s} {'gamma':>8s} {'spread':>8s} {'eps_nx':>8s} {'beta':>8s} {'E [mJ]':>8s}")
for lattice, r in results.items():
    for (where, who), b in r.items():
        print(f"{lattice:12s} {where:9s} {who:8s} {b['gamma']:8.1f} {b['spread']:8.2%} {b['eps_nx']:8.3f} "
              f"{b['beta']:8.2f} {b['energy']:8.3f}")
    lost = r[("entrance", "driver")]["energy"] - r[("exit", "driver")]["energy"]
    gained = r[("exit", "witness")]["energy"] - r[("entrance", "witness")]["energy"]
    print(f"{lattice:12s} driver lost {lost:.2f} mJ, witness gained {gained:.2f} mJ ({gained / lost:.0%})\n")

# --- Checks ----------------------------------------------------------------
problems = []
for lattice, r in results.items():
    lost = r[("entrance", "driver")]["energy"] - r[("exit", "driver")]["energy"]
    gained = r[("exit", "witness")]["energy"] - r[("entrance", "witness")]["energy"]
    if not 0 < gained < lost:
        problems.append(f"{lattice}: driver lost {lost:.3f} mJ, witness gained {gained:.3f} mJ")
    w, s = r[("exit", "witness")], r[("screen", "witness")]
    got = (w["gamma"], w["spread"], w["eps_nx"], w["beta"], s["eps_nx"])
    if max(abs(g / ref - 1) for g, ref in zip(got, README[lattice])) > 0.01:
        problems.append(f"{lattice}: {np.round(got, 4)} against the README's {README[lattice]}")
single, ramps = results["pwfa_single"], results["pwfa_ramps"]
if ramps[("exit", "witness")]["beta"] / single[("exit", "witness")]["beta"] < 4:
    problems.append("the ramps grew beta at the exit less than 4x")
if ramps[("screen", "witness")]["eps_nx"] >= single[("screen", "witness")]["eps_nx"]:
    problems.append("the ramps did not reduce the emittance growth in the drift")
assert not problems, "\n".join(problems)
print("OK: the driver gives energy to the witness, the README's witness table is reproduced, and the ramps "
      "grow beta and cut the drift's emittance growth")

## 30 A start-to-end FEL: elegant into Genesis 4

JFEL's linac in elegant (wakes and CSR on), then Genesis 4 steady-state with a 1 kW seed. The Genesis settings are in the line's `fel:` block. The wavelength is the resonance $\lambda = \lambda_u (1 + K^2) / (2\gamma^2)$ = 2.38 µm. The gain length comes out 1.5× Ming Xie's estimate, computed in the script from the beam at the undulator: Ming Xie treats the beam as round, and after CSR $\varepsilon_{n,x}$ is ~5× $\varepsilon_{n,y}$. The time-dependent SASE line `FEL_TD` takes ~4 min and isn't run.

In [ ]:
__file__ = "30_fel_genesis.py"  # python and %run set this; a notebook cell does not
"""
30 - A start-to-end FEL: elegant into Genesis 4
===============================================

Shows: a Genesis 4 line after a tracking line. JFEL's Lattices/jfel_fel.def
takes the saved injector beam through the linac in elegant (structure wakes
and CSR on) to the undulator entrance, then hands that beam to Genesis 4,
steady-state (FEL_ST, a 1 kW seed, seconds) or time-dependent (FEL_TD, SASE,
~4 min; not run here). Genesis' settings are in each line's `fel:` block.
SIMBA writes Genesis' input from the beam and the LAURA undulator; Genesis'
own output is FEL_ST.out.h5 in the run directory.

Run:    python 05_advanced/30_fel_genesis.py
Tier:   B
Needs:  SIMBA, LAURA, laura-lattices (JFEL), h5py; docker and
        ghcr.io/astec-stfc/simcodes-docker:master (elegant and Genesis).
        Prints SKIP and exits 0 without them.
Output: output/30_fel_genesis/ (FEL_ST.out.h5 is Genesis' output).

Units: wavelength in um; lengths in m; power in W; current in A.

The undulator (JFEL-FEL-MAG-WIGG-01) is helical, K = 8.52, period 82 mm,
ten segments. The Ming Xie gain length is computed here from the beam at
the undulator entrance: peak current, mean gamma, rms spread, the
geometric-mean size and emittance (Xie, NIM A 445 (2000) 59; the fit and
its 19 coefficients as in laura-lattices' JFEL/Scripts/run_fel.py).

Checks:
- Wavelength: Genesis' reference wavelength equals lambda_u (1 + K^2) /
  (2 gamma^2) with the beam's mean gamma, to 1e-3 (measured 8e-7: Genesis
  takes its reference from the same gamma, so this checks the hand-over).
- Gain length, fitted to Genesis' power between 1e-4 and 1e-2 of its
  maximum: longer than the 1D gain length (0.21 m; 3D effects only add to
  it), and 1x to 2x Ming Xie's (measured 0.757 against 0.498 m, 1.52x).
  Genesis is longer because Ming Xie treats the beam as round with
  averaged parameters, and after CSR eps_nx is ~5x eps_ny (JFEL README).
- The JFEL README's steady-state row: lambda 2.380 um to 1e-3, L_g 0.76 m
  and Ming Xie 0.50 m to 2%, 1.7 GW at the exit to 5%. The run repeats
  exactly, so these are the README's two-digit rounding (up to 3% for
  1.7) plus version drift. Measured: 0.4%, 0.4%, 1.5%.
"""

import os
import shutil
import subprocess
from pathlib import Path

import h5py
import numpy as np
import simba.Framework as fw
import simba.Modules.Beams as rbf
from laura_lattices import JFEL

OUT = Path(__file__).resolve().parents[1] / "output" / Path(__file__).stem
JFEL_DIR = os.path.dirname(JFEL.__file__)
IMAGE = "ghcr.io/astec-stfc/simcodes-docker:master"
ENTRANCE = "JFEL-FEL-SIM-MARK-01"
MC2, IA = 0.51099895e6, 17045.0  # eV; Alfven current [A]
XIE = [0.45, 0.57, 0.55, 1.6, 3, 2, 0.35, 2.9, 2.4, 51, 0.95, 3, 5.4, 0.7, 1.9, 1140, 2.2, 2.9, 3.2]
README = {"lambda": 2.380e-6, "Lg": 0.76, "Lg_xie": 0.50, "P_end": 1.7e9}  # JFEL README, steady-state


def docker_ready():
    return bool(shutil.which("docker")) and subprocess.run(
        ["docker", "image", "inspect", IMAGE], capture_output=True).returncode == 0


def ming_xie(beam, period, K):
    """Ming Xie's 3D gain length for `beam` in a helical undulator; also rho, the 1D gain length, current."""
    q, t = np.abs(np.asarray(beam.Particles.charge.val)), np.asarray(beam.t.val)
    counts, edges = np.histogram(t, bins=max(int(np.sqrt(len(t))), 20), weights=q)
    current = np.convolve(counts / np.diff(edges), np.ones(3) / 3, "same").max()
    gamma = np.mean(beam.cp.val) / MC2
    spread = np.std(beam.cp.val) / np.mean(beam.cp.val)
    sigma = np.sqrt(np.std(beam.x.val) * np.std(beam.y.val))
    eps = np.sqrt(beam.Particles.emittance.enx.val * beam.Particles.emittance.eny.val) / gamma
    ku, lam = 2 * np.pi / period, period * (1 + K ** 2) / (2 * gamma ** 2)
    rho = (current / IA * 2 * K ** 2 / (16 * gamma ** 3 * sigma ** 2 * ku ** 2)) ** (1 / 3)  # helical: [JJ] = 1
    lg1d = period / (4 * np.pi * np.sqrt(3) * rho)
    d, e, g = lg1d * lam / (4 * np.pi * sigma ** 2), lg1d * 4 * np.pi * eps / (sigma ** 2 / eps * lam), 4 * np.pi * lg1d * spread / period
    a = XIE
    eta = (a[0] * d ** a[1] + a[2] * e ** a[3] + a[4] * g ** a[5] + a[6] * e ** a[7] * g ** a[8]
           + a[9] * d ** a[10] * g ** a[11] + a[12] * d ** a[13] * e ** a[14] + a[15] * d ** a[16] * e ** a[17] * g ** a[18])
    return {"Lg": lg1d * (1 + eta), "Lg1d": lg1d, "rho": rho, "current": current, "gamma": gamma}


if not docker_ready():
    print(f"SKIP: needs docker and {IMAGE} (elegant and Genesis)")
else:  # not sys.exit(0): in a notebook that is an error and stops "Run All"
    f = fw.Framework(master_lattice=JFEL_DIR + "/", directory=str(OUT), clean=True, verbose=False,
                     container_runtime="docker")
    f.loadSettings("Lattices/jfel_fel.def")
    f.track(files=["Linac", "FEL_ST"], check_lattice=False)

    undulator = f.getElement("JFEL-FEL-MAG-WIGG-01").magnetic
    period, K = float(undulator.period), float(undulator.strength)
    assert undulator.helical, "the formulas here are for a helical undulator"
    xie = ming_xie(rbf.load_file(str(OUT / f"{ENTRANCE}.openpmd.hdf5")), period, K)
    lam_formula = period * (1 + K ** 2) / (2 * xie["gamma"] ** 2)

    with h5py.File(OUT / "FEL_ST.out.h5") as h:
        z, power = h["Lattice/zplot"][()], h["Field/power"][()][:, 0]
        lam_genesis = float(h["Global/lambdaref"][()][0])
    peak = int(np.argmax(power))
    exponential = (power > 1e-4 * power[peak]) & (power < 1e-2 * power[peak]) & (np.arange(len(z)) < peak)
    lg = 1 / np.polyfit(z[exponential], np.log(power[exponential]), 1)[0]

    print(f"\nbeam at the undulator: I_peak {xie['current']:.0f} A, gamma {xie['gamma']:.1f}, rho {xie['rho']:.4f}")
    print(f"wavelength: Genesis {lam_genesis * 1e6:.4f} um, formula {lam_formula * 1e6:.4f} um "
          f"({lam_genesis / lam_formula - 1:+.1e}); K {K}, period {period} m")
    print(f"gain length: Genesis {lg:.3f} m, Ming Xie {xie['Lg']:.3f} m, 1D {xie['Lg1d']:.3f} m "
          f"(Genesis / Ming Xie = {lg / xie['Lg']:.2f})")
    print(f"power: {power[peak]:.3e} W at most (z = {z[peak]:.1f} m), {power[-1]:.3e} W at the exit")
    print(f"README: lambda {README['lambda'] * 1e6} um, L_g {README['Lg']} m, Ming Xie {README['Lg_xie']} m, "
          f"{README['P_end']:.1e} W at the exit")

    # --- Checks ------------------------------------------------------------
    problems = []
    if abs(lam_genesis / lam_formula - 1) > 1e-3:
        problems.append(f"wavelength {lam_genesis:.5e}, formula {lam_formula:.5e}")
    if lg < xie["Lg1d"]:
        problems.append(f"gain length {lg:.3f} m shorter than the 1D one {xie['Lg1d']:.3f} m")
    if not 1 <= lg / xie["Lg"] <= 2:
        problems.append(f"gain length {lg:.3f} m, {lg / xie['Lg']:.2f}x Ming Xie's")
    for got, key, limit in ((lam_genesis, "lambda", 1e-3), (lg, "Lg", 0.02), (xie["Lg"], "Lg_xie", 0.02),
                            (power[-1], "P_end", 0.05)):
        if abs(got / README[key] - 1) > limit:
            problems.append(f"{key}: {got:.4g}, README {README[key]}")
    assert not problems, "\n".join(problems)
    print("\nOK: the resonant wavelength, a gain length between 1x and 2x Ming Xie's, and the README's row")

## 31 Secondaries in BDSIM

LAURA `Target` and `Crystal` elements, tracked in BDSIM (Geant4). The line's `tracking:` block names the physics list and the secondaries to keep, and they come out as their own species in the same beam file, `beam.others["photon"]` and so on. A tungsten target is checked against the mean radiation loss $E_0 e^{-t/X_0}$. A converter's positrons go on to Xsuite on their own (`input: {species: positron}`). A bent crystal channels ~9% of the electrons through its 0.9 mrad bend. Geant4's seed changes every run, so the yields vary by a few percent.

In [ ]:
__file__ = "31_secondaries_bdsim.py"  # python and %run set this; a notebook cell does not
"""
31 - Secondaries in BDSIM: a target, a converter, a bent crystal
================================================================

Shows: beam-matter lines in BDSIM (Geant4) from LAURA `Target` and
`Crystal` elements. A line's `tracking:` block picks the Geant4
`physics_list`, the `secondaries` to keep and extra `samplers`; the
secondaries come out as their own species in the same openPMD file,
`beam.others[species]`, next to the primaries. A next line can take one
species on to another code (`input: {species: positron}`).

Run:    python 05_advanced/31_secondaries_bdsim.py
Tier:   C
Needs:  SIMBA, LAURA, laura-lattices (SECONDARY), Xsuite; BDSIM on PATH
        with ROOT's Python bindings and pybdsim (source ROOT's thisroot.sh
        and BDSIM's bdsim.sh first). Prints SKIP and exits 0 without them.
Output: output/31_secondaries_bdsim/<lattice>/.

Units: yields per incoming electron; mean energies in MeV; angles in mrad.

laura-lattices' three SECONDARY lines, 2000 macroparticles each (~1 min in
all):
- brems_target: 100 MeV e- through 1 mm of tungsten (0.29 X0), sampled at
  the target's exit;
- positron_conversion: 1 GeV e- shower in 3.5 mm of tungsten (1 X0); the
  positrons alone go on to a screen in Xsuite;
- crystal_channelling: 855 MeV e- on 30 um of bent Si(220), 0.9 mrad.

Geant4's seed changes every run, so each check allows for the statistics
of 2000 macroparticles.

Checks (measured values from one run, against SIMBA and LAURA main):
- brems_target, at the target's exit: the electrons' mean energy is
  E0 exp(-t/X0), 75.2 MeV, to 3% (the formula is the mean radiation loss;
  measured 75.3). Electrons per electron 0.989 to 0.01 (README; a binomial
  sigma is 0.002; measured 0.987). Photons per electron 2.03 and positrons
  0.038, the README's SIMBA row, to 5 sigma of a Poisson count over 2000
  (+-0.16, +-0.022; measured 2.08, 0.033).
- positron_conversion, at the converter's exit: positrons per electron in
  the README's range over runs, 0.84-0.87, widened by its quoted seed
  spread of +-0.02 (0.82-0.89; measured 0.834); photons 11.7-11.9 the same
  way, 0.2 for the spread (11.5-12.1; measured 11.89). The Xsuite screen
  gets the positrons, and the photons carried through untracked.
- crystal_channelling: the fraction within 0.15 mrad of the 0.9 mrad bend
  (channelled through the whole crystal) is 0.085 to 5 sigma, as the
  yields (+-0.035). 0.085 and sigma = 0.0077 are the mean and spread of
  15 runs, 0.066-0.097 (the binomial sigma for 2000 is 0.0062; the 0.066
  run, 2.4 sigma low, failed an earlier 3-sigma band). The README has
  0.093 for SIMBA and 0.082 for its hand-written gmad, one run each.
  Amorphous scattering alone gives 0.004.
"""

import importlib.util
import os
import shutil
from pathlib import Path

import numpy as np
import simba.Framework as fw
import simba.Modules.Beams as rbf
from laura_lattices import SECONDARY

OUT = Path(__file__).resolve().parents[1] / "output" / Path(__file__).stem
SECONDARY_DIR = os.path.dirname(SECONDARY.__file__)
E = 1.602176634e-19
X0_W = 3.504e-3  # tungsten radiation length [m] (PDG)


def species(path):
    """{species: (real particles, <E> [MeV], x' [mrad] array)} in a SIMBA beam file."""
    beam = rbf.load_file(str(path))
    out = {}
    for name, b in {beam.species: beam, **beam.others}.items():
        n = np.broadcast_to(np.asarray(b.Particles.nmacro.val, dtype=float), (len(b.x),))
        out[name] = (n.sum(), np.average(np.asarray(b.energy.val) / 1e6, weights=n), np.asarray(b.xp.val) * 1e3)
    return out


def run(lattice):
    """Track `lattice`; the species at each sampler and line end, per incoming electron."""
    directory = OUT / lattice
    f = fw.Framework(master_lattice=SECONDARY_DIR + "/", directory=str(directory), clean=True, verbose=False)
    f.loadSettings(f"Lattices/{lattice}.def")
    f.track()
    lines = [line for name, line in f.latticeObjects.items() if name != "generator"]
    stem = lines[0].file_block["input"]["particle_definition"].replace("$master_lattice$", SECONDARY_DIR + "/")
    electrons = abs(float(rbf.load_file(stem + ".openpmd.hdf5").Particles.total_charge)) / E
    points = {}
    for line in lines:
        for name in list((line.file_block.get("tracking") or {}).get("samplers") or []) + [line.end]:
            points[name] = {s: (n / electrons, e, xp) for s, (n, e, xp) in species(directory / f"{name}.openpmd.hdf5").items()}
    return electrons, points


if not shutil.which("bdsim") or importlib.util.find_spec("pybdsim") is None or importlib.util.find_spec("ROOT") is None:
    print("SKIP: needs bdsim on PATH, ROOT's Python bindings and pybdsim (source thisroot.sh and bdsim.sh)")
else:  # not sys.exit(0): in a notebook that is an error and stops "Run All"
    results = {lattice: run(lattice) for lattice in ("brems_target", "positron_conversion", "crystal_channelling")}
    for lattice, (electrons, points) in results.items():
        print(f"\n{lattice} (incoming: {electrons:.3g} electrons, as 2000 macroparticles)")
        for point, by_species in points.items():
            for s, (n, e, xp) in by_species.items():
                print(f"  {point:22s} {s:9s} {n:7.3f} per e-   <E> {e:8.2f} MeV   x' median {np.median(xp):7.3f} mrad")

    brems = results["brems_target"][1]["SEC-BREMS-TGT-01"]
    conv_points = results["positron_conversion"][1]
    conversion, screen = conv_points["SEC-CONV-SIM-END-01"], conv_points["SEC-CONV-DIA-SCR-01"]
    crystal_xp = results["crystal_channelling"][1]["SEC-CRY-DIA-SCR-01"]["electron"][2]
    channelled = np.mean(np.abs(crystal_xp - 0.9) < 0.15)
    expected_energy = 100.0 * np.exp(-1e-3 / X0_W)
    print(f"\nbrems: electrons' <E> {brems['electron'][1]:.2f} MeV, E0 exp(-t/X0) {expected_energy:.2f} MeV")
    print(f"crystal: channelled fraction {channelled:.3f} (mean of 15 runs 0.085; README 0.093)")

    # --- Checks ------------------------------------------------------------
    problems = []
    checks = [  # (what, got, low, high)
        ("brems e- <E> [MeV]", brems["electron"][1], expected_energy * 0.97, expected_energy * 1.03),
        ("brems e- per e-", brems["electron"][0], 0.979, 0.999),
        ("brems photons per e-", brems["photon"][0], 2.03 - 0.16, 2.03 + 0.16),
        ("brems e+ per e-", brems.get("positron", (0,))[0], 0.038 - 0.022, 0.038 + 0.022),
        ("conversion e+ per e-", conversion["positron"][0], 0.82, 0.89),
        ("conversion photons per e-", conversion["photon"][0], 11.5, 12.1),
        ("crystal channelled", channelled, 0.085 - 0.035, 0.085 + 0.035),
    ]
    for what, got, low, high in checks:
        if not low <= got <= high:
            problems.append(f"{what}: {got:.4f}, expected {low:.4f} to {high:.4f}")
    tracked = [s for s in screen if s != "photon"]  # photons are carried through untracked
    if tracked != ["positron"]:
        problems.append(f"the Xsuite screen has {tracked}, expected the positrons alone")
    assert not problems, "\n".join(problems)
    print("\nOK: the radiation loss, the yields and the channelled fraction are as expected")

## 32 Bmad, and Bmad multipass

28's ERL in Bmad, twice: flattened, as the other codes track it, and with `multipass: native`, where LAURA writes Bmad's own multipass. Each cavity is one lord with a slave per pass, and the return pass's phase is `phi0_multipass` on the slave. The two agree to $10^{-5}$. Needs `ACC_ROOT_DIR` set before Jupyter starts.

In [ ]:
__file__ = "32_bmad_multipass.py"  # python and %run set this; a notebook cell does not
"""
32 - Bmad: the ERL flattened, and as Bmad multipass
===================================================

Shows: Bmad through SIMBA (pytao and libtao), and LAURA's Bmad multipass
export. Every other code tracks a multipass layout flattened, with a copy
of each cavity per pass (28). Bmad can hold the shared hardware itself:
`multipass: native` on a line (ERL/Lattices/erl_bmad_multipass.def) writes
each cavity once, as a multipass lord with one slave per pass, and the
second pass's phase per slave:

    LIN: line[multipass] = (..., ERL_LIN_RF_CAV_01, ...)
    ERL_LIN_RF_CAV_01\\2[phi0_multipass] = -0.5

Run:    python 05_advanced/32_bmad_multipass.py
Tier:   C
Needs:  SIMBA, LAURA, laura-lattices (ERL); Bmad: pytao and
        $ACC_ROOT_DIR/production/lib/libtao.so (export ACC_ROOT_DIR, and put
        $ACC_ROOT_DIR/production/lib first on LD_LIBRARY_PATH; README,
        troubleshooting). Prints SKIP and exits 0 without them.
Output: output/32_bmad_multipass/<flattened|native>/ (ERL.bmad is the
        lattice Bmad read).

Units: momentum in MeV/c; spread relative, rms; emittance (normalised) in
um; sizes in mm; sigma_t in ps.

Both runs are single-particle: SIMBA lines start with CSR and wakes on, so
they are switched off here. Each takes ~10 s.

Checks:
- Native multipass equals the flattened lattice at the dump screen, in
  momentum, spread, emittances, sizes and sigma_t, to 1e-4 (measured
  9e-6; the README says "identical"; the two lattices are sliced
  differently).
- Flattened, the ERL README's Bmad column: 10.000, 100.000, 10.000 MeV/c
  (to 1e-4; measured 2e-6), top and dump spreads 1.98e-4, 1.00e-3 and the
  dump's sigma_x / sigma_y 0.579 / 0.580 mm, sigma_t 2.01 ps (to 1%, the
  README's rounding plus version drift).
"""

import importlib.util
import os
from pathlib import Path

import numpy as np
import simba.Framework as fw
import simba.Modules.Beams as rbf
from laura_lattices import ERL

OUT = Path(__file__).resolve().parents[1] / "output" / Path(__file__).stem
ERL_DIR = os.path.dirname(ERL.__file__)
LIBTAO = os.path.join(os.environ.get("ACC_ROOT_DIR", ""), "production", "lib", "libtao.so")
SCREEN = "ERL-DMP-DIA-SCR-01"
QUANTITIES = ("p", "spread", "eps_nx", "eps_ny", "sigma_x", "sigma_y", "sigma_t")
README = {"injected": {"p": 10.000}, "top": {"p": 100.000, "spread": 1.98e-4},  # ERL README, Bmad column
          "dump": {"p": 10.000, "spread": 1.00e-3, "sigma_x": 0.579, "sigma_y": 0.580, "sigma_t": 2.01}}
POINTS = {"injected": "ERL-LIN-SIM-START-01.1", "top": "ERL-LIN-SIM-END-01.1", "dump": "ERL-LIN-SIM-END-01.2"}


def summary(path):
    b = rbf.load_file(str(path))
    cp = b.cp.val
    return dict(zip(QUANTITIES, (np.mean(cp) / 1e6, np.std(cp) / np.mean(cp), b.Particles.emittance.enx.val * 1e6,
                                 b.Particles.emittance.eny.val * 1e6, np.std(b.x.val) * 1e3, np.std(b.y.val) * 1e3,
                                 np.std(b.t.val) * 1e12)))


def track(native):
    directory = OUT / ("native" if native else "flattened")
    f = fw.Framework(master_lattice=ERL_DIR + "/", directory=str(directory), clean=True, verbose=False)
    f.executables.define_tao_command(location=[LIBTAO])
    f.loadSettings("Lattices/erl_bmad_multipass.def" if native else "Lattices/erl.def")
    for name in [n for n in f.latticeObjects if n != "generator"]:
        if not native:
            f.change_Lattice_Code(name, "bmad")
        f[name].csr_enable = f[name].wakefield_enable = False
    f.track(check_lattice=False)
    return directory


if not os.path.isfile(LIBTAO) or importlib.util.find_spec("pytao") is None:
    print("SKIP: needs pytao and $ACC_ROOT_DIR/production/lib/libtao.so (export ACC_ROOT_DIR)")
else:  # not sys.exit(0): in a notebook that is an error and stops "Run All"
    flattened, native = track(False), track(True)
    print("Bmad multipass in the native lattice (ERL.bmad):")
    for line in (native / "ERL.bmad").read_text().splitlines():
        if "line[multipass]" in line or "RF_CAV_01\\" in line:
            print("   ", line[:100] + (" ..." if len(line) > 100 else ""))

    at = {"flattened": summary(flattened / f"{SCREEN}.openpmd.hdf5"), "native": summary(native / f"{SCREEN}.openpmd.hdf5")}
    points = {k: summary(flattened / f"{p}.openpmd.hdf5") for k, p in POINTS.items()}
    print(f"\n{'':22s}" + "".join(f"{q:>10s}" for q in QUANTITIES))
    for name, s in [(f"screen, {k}", v) for k, v in at.items()] + [(f"flattened, {k}", v) for k, v in points.items()]:
        print(f"{name:22s}" + "".join(f"{s[q]:10.4g}" for q in QUANTITIES))

    # --- Checks ------------------------------------------------------------
    problems = []
    worst = max(abs(at["native"][q] / at["flattened"][q] - 1) for q in QUANTITIES)
    print(f"\nnative against flattened at the screen: largest relative difference {worst:.1e}")
    if worst > 1e-4:
        problems.append(f"native multipass differs from the flattened lattice by {worst:.1e}")
    for point, refs in README.items():
        for q, ref in refs.items():
            limit = 1e-4 if q == "p" else 0.01
            if abs(points[point][q] / ref - 1) > limit:
                problems.append(f"{point} {q}: {points[point][q]:.5g}, README {ref}")
    assert not problems, "\n".join(problems)
    print("OK: Bmad multipass equals the flattened lattice, and the README's Bmad column is reproduced")